# UKCP18 CPM rainfall to HEC-RAS DSS

Convert regional UKCP18 CPM rainfall to gridded DSS precipitation for HEC-RAS rain-on-grid models. Output rainfall uses MM and PER-CUM for each input accumulation interval.

In [ ]:
from pathlib import Path
import json
from dss_pipeline import main


## Configuration

Set the input and output paths. Set `boundary` to a polygon shapefile and `name` to an output label. The example uses the broad Forth region from the UK river-basin regions dataset. Set `choose_boundary=True` to browse for another shapefile. Date-only end values include the whole day; datetime end values are exclusive.

For native rotated grids, add `target_crs='EPSG:27700'` and `target_cell_size=2200`. Set `inspect=True` to display source metadata without writing output.

Set `boundary=None`, keep `choose_boundary=False` and leave `boundary_dir` unset to use the full input grid.

DSS output requires the HEC-DSSVue `Jython.exe` path and a precipitation grid template. Set `prepare_only=True` to prepare rainfall data without writing DSS.

In [ ]:
CONFIG = {'input_dir': 'D:/climate/ukcp18-cpm',
 'input_pattern': 'pr*.nc',
 'output_dir': 'D:/hec-ras/precipitation',
 'variable': 'pr',
 'boundary': 'D:/boundaries/forth.shp',
 'choose_boundary': False,
 'name': 'forth',
 'jython_exe': 'C:/HEC/HEC-DSSVue/Jython.exe',
 'template_dss': 'examples/output_dss/pr_forth_20y_01_20191101_20391231.dss',
 'window': [['01', '2019-11-01', '2039-12-31'],
            ['02', '2039-11-01', '2059-12-31'],
            ['03', '2059-11-01', '2080-11-30']],
 'prepare_only': False}

RUN = False


In [ ]:
def cli_arguments(config):
    arguments = []
    for key, value in config.items():
        flag = "--" + key.replace("_", "-")
        if key == "window":
            for row in value:
                arguments.extend([flag, *map(str, row)])
        elif key == "select":
            for item in value:
                arguments.extend([flag, str(item)])
        elif isinstance(value, bool):
            if value:
                arguments.append(flag)
        elif isinstance(value, list):
            arguments.extend([flag, *map(str, value)])
        elif value is not None:
            arguments.extend([flag, str(value)])
    return arguments


## Run

Set `RUN=True` after editing the configuration. Source files are read-only. Use a new output directory or set `overwrite=True` to replace an existing output.

In [ ]:
if RUN:
    jobs = main('cpm', cli_arguments(CONFIG), clip=False)
    for metadata_path in jobs:
        metadata = json.loads(Path(metadata_path).read_text(encoding='ascii'))
        print({
            'start': metadata['start'],
            'end_exclusive': metadata['end_exclusive'],
            'intervals': metadata['record_count'],
            'cell_size_m': metadata['cell_size'],
        })
else:
    print('Edit CONFIG and set RUN=True to start.')
